# Prepare Kaggle Dataset From Google Drive (DOTA)

???? ??????? ????????? **?????? DOTA** ? Kaggle Dataset ????? Kaggle API.

????????? ????????? ???????? ????? ? Google Drive:
- `images/train`, `images/val`
- `labels/train`, `labels/val`


In [ ]:
# Step 0: install Kaggle API.
import sys
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
print('[OK] kaggle installed')

In [ ]:
# Step 1: mount Drive and set paths.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path('/content/drive/MyDrive')
WORK_ROOT = Path('/content/kaggle_dataset_build/dota')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_ROOT = DRIVE_ROOT / 'datasets' / 'dota_yolo'
print('SOURCE_ROOT =', SOURCE_ROOT)

In [ ]:
# Step 2: configure Kaggle account and dataset slug.
from pathlib import Path
import os, json, shutil

KAGGLE_JSON_FROM_DRIVE = DRIVE_ROOT / 'kaggle' / 'kaggle.json'
KAGGLE_JSON_LOCAL = Path('/content/kaggle.json')

if KAGGLE_JSON_FROM_DRIVE.exists():
    src_token = KAGGLE_JSON_FROM_DRIVE
elif KAGGLE_JSON_LOCAL.exists():
    src_token = KAGGLE_JSON_LOCAL
else:
    raise FileNotFoundError('kaggle.json not found. Place it in MyDrive/kaggle or /content.')

Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
dst_token = Path('/root/.kaggle/kaggle.json')
shutil.copy2(src_token, dst_token)
os.chmod(dst_token, 0o600)

token = json.loads(dst_token.read_text(encoding='utf-8'))
KAGGLE_USERNAME = token.get('username')
if not KAGGLE_USERNAME:
    raise RuntimeError('Cannot read username from kaggle.json')

DATASET_TITLE = 'small-objects-dota-yolo'
DATASET_ID = f'{KAGGLE_USERNAME}/{DATASET_TITLE}'
print('[OK] Kaggle user:', KAGGLE_USERNAME)
print('DATASET_ID =', DATASET_ID)

In [ ]:
# Step 3: validate source and prepare upload folder (strict checks).
from pathlib import Path
import shutil, json

KEEP_EXTRA_FILES = ['dataset.yaml', 'data.yaml', 'dota.yaml', 'README.md']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


def count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def count_labels(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob('*.txt') if p.is_file())


def is_ready_yolo(root: Path) -> bool:
    req = [
        root / 'images' / 'train',
        root / 'images' / 'val',
        root / 'labels' / 'train',
        root / 'labels' / 'val',
    ]
    return all(p.exists() for p in req) and count_images(root / 'images' / 'train') > 0 and count_images(root / 'images' / 'val') > 0


if not is_ready_yolo(SOURCE_ROOT):
    raise FileNotFoundError(
        f'Source dataset is not ready: {SOURCE_ROOT}. '        'Expected images/train, images/val, labels/train, labels/val with non-empty images.'
    )

UPLOAD_DIR = WORK_ROOT / 'upload'
if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

for rel in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    src = SOURCE_ROOT / rel
    dst = UPLOAD_DIR / rel
    shutil.copytree(src, dst)

for name in KEEP_EXTRA_FILES:
    src = SOURCE_ROOT / name
    if src.exists() and src.is_file():
        shutil.copy2(src, UPLOAD_DIR / name)

meta = {
    'title': DATASET_TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}
(UPLOAD_DIR / 'dataset-metadata.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

n_train = count_images(UPLOAD_DIR / 'images' / 'train')
n_val = count_images(UPLOAD_DIR / 'images' / 'val')
n_lbl_train = count_labels(UPLOAD_DIR / 'labels' / 'train')
n_lbl_val = count_labels(UPLOAD_DIR / 'labels' / 'val')

print('[OK] upload folder prepared:', UPLOAD_DIR)
print('train images:', n_train)
print('val images:', n_val)
print('train labels:', n_lbl_train)
print('val labels:', n_lbl_val)

assert n_train > 0 and n_val > 0, 'No images copied to upload dir'
assert n_lbl_train > 0 and n_lbl_val > 0, 'No labels copied to upload dir'

print('
Top-level upload files/folders:')
for p in sorted(UPLOAD_DIR.iterdir()):
    print(' -', p.name)


In [ ]:
# Step 4: publish dataset (create or new version) + verify remote file list.
import subprocess

PUBLIC = True


def kaggle_dataset_exists(dataset_id: str) -> bool:
    result = subprocess.run(['kaggle', 'datasets', 'status', dataset_id], capture_output=True, text=True)
    return result.returncode == 0


def publish(upload_dir: str, dataset_id: str, message: str):
    if kaggle_dataset_exists(dataset_id):
        cmd = ['kaggle', 'datasets', 'version', '-p', upload_dir, '-m', message, '--dir-mode', 'zip']
        print('[INFO] Version command:', ' '.join(cmd))
    else:
        cmd = ['kaggle', 'datasets', 'create', '-p', upload_dir, '--dir-mode', 'zip']
        if PUBLIC:
            cmd.append('--public')
        print('[INFO] Create command:', ' '.join(cmd))
    subprocess.run(cmd, check=True)


publish(str(UPLOAD_DIR), DATASET_ID, message='Upload full DOTA YOLO structure')
print('[OK] Publish finished')

print('
[INFO] Remote Kaggle file list:')
result = subprocess.run(['kaggle', 'datasets', 'files', DATASET_ID], check=True, capture_output=True, text=True)
print(result.stdout)


In [ ]:
# Step 5: final URL and expected Kaggle input path.
print('Kaggle URL:', f'https://www.kaggle.com/datasets/{DATASET_ID}')
print('Expected input path in Kaggle notebook:', f'/kaggle/input/{DATASET_TITLE}')